# Quickstart: one question, four options, one token

This notebook shows the mechanics of a Jev-style decision on Cloudera AI, step by step.
You give the model a piece of text (the **state**), a **question**, and a fixed list of
**options**. Instead of writing an answer, the model returns a probability for each
option. (The field names follow the Jev interface, as
[SemIf-OpenJev](https://github.com/TheoLeeCJ/SemIf-OpenJev) does.)

The example: deciding what a short news story is about: World, Sports, Business or
Sci/Tech. The model is a standard deployment on Cloudera AI Inference Service (here
Qwen/Qwen2.5-7B-Instruct, set by `CAI_MODEL`), called through its OpenAI-compatible
API. The saved outputs come from one run on 2026-10-05.

## 1. Connect

The endpoint is configured only by environment variables, so the same notebook runs
on public or private cloud. This prints whether each is set (never the address) and
builds an `openai` client. The client keeps a copy of each request body, so we can show
exactly what was sent (never the headers, which carry the token).

In [1]:
import json
import math
import os
import sys
from pathlib import Path

import httpx
import pandas as pd
from openai import OpenAI

def find_project_root():
    here = Path.cwd()
    home = Path("/home/cdsw")
    candidates = [here, *here.parents, home]
    if home.is_dir():
        candidates += sorted(p for p in home.iterdir() if p.is_dir())
    for c in candidates:
        if (c / "jev" / "client.py").is_file() and (c / "notebooks" / "sample_texts.jsonl").is_file():
            return c
    raise RuntimeError("Could not find the project folder.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from jev.client import load_config   # reads CAI_BASE_URL, CAI_MODEL, CAI_TOKEN, CAI_CA_BUNDLE

for name in ("CAI_BASE_URL", "CAI_MODEL"):
    print(f"{name}: {'set' if os.environ.get(name) else 'NOT SET'}")
cfg = load_config()

sent = {}
def keep_request_body(request):
    sent["body"] = json.loads(request.content)

client = OpenAI(
    base_url=cfg["base"],
    api_key=cfg["token"],
    http_client=httpx.Client(verify=cfg["verify"], timeout=60,
                             event_hooks={"request": [keep_request_body]}),
)
print("Client ready")

CAI_BASE_URL: set
CAI_MODEL: set
Client ready


## 2. The data

Ten short news-style texts we wrote for this notebook, each with the topic we had in
mind. Two are deliberately mixed (the business side of a sports story; a science
project funded by governments); step 8 shows how the model handles them.

**Optional: AG News.** Set `USE_AG_NEWS = True` to fetch rows of the public AG News
dataset (the same four topics) at run time from the Hugging Face datasets API. The
rows are fetched by you and are not part of this repository; the dataset's license is
listed as unknown, so check its terms before you use or publish it.

In [2]:
USE_AG_NEWS = False
AG_NEWS_LABELS = ["world", "sports", "business", "sci_tech"]   # AG News label order

def fetch_ag_news(n=10, offset=0):
    r = httpx.get("https://datasets-server.huggingface.co/rows",
                  params={"dataset": "fancyzhx/ag_news", "config": "default", "split": "test",
                          "offset": offset, "length": n}, timeout=30)
    r.raise_for_status()
    return [{"id": f"ag-{offset + i}", "text": row["row"]["text"],
             "label": AG_NEWS_LABELS[row["row"]["label"]]}
            for i, row in enumerate(r.json()["rows"])]

if USE_AG_NEWS:
    texts = fetch_ag_news()
else:
    with open("notebooks/sample_texts.jsonl") as f:
        texts = [json.loads(line) for line in f]

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(texts)[["id", "label", "text"]]

,id,label,text
0,text-01,world,Leaders of three neighbouring countries met in the coastal city of Port Avalen to sign a water-sharing treaty after ...
1,text-02,sports,"The Riverton Hawks came back from two goals down to win the regional cup final 3-2, with the winning goal scored in ..."
2,text-03,business,A mid-sized furniture maker reported a 12% rise in quarterly profit and said it would open two new factories next ye...
3,text-04,sci_tech,"Researchers built a battery cell from sodium and iron that kept 90% of its capacity after 5,000 charge cycles in lab..."
4,text-05,world,"Thousands of people were moved to shelters as floods spread across the northern provinces, and the government asked ..."
5,text-06,sports,"The national tennis champion withdrew from next week's open with a wrist injury, ending her hopes of a third straigh..."
6,text-07,business,Central bank officials held interest rates steady and warned that shipping costs could push consumer prices higher t...
7,text-08,sci_tech,A new open-source tool lets programmers find memory leaks in their code by replaying a program's execution step by s...
8,text-09,business,"The owners of the Riverton Hawks agreed to sell the football club to an investment group, valuing the team and its s..."
9,text-10,sci_tech,"Two governments agreed to jointly fund a weather satellite, which will be built by a university lab and launched in ..."


## 3. The decision, as plain data

One text, one question, four options. Each option has an id (what your code gets
back) and a description (what the model reads).

In [3]:
state = texts[0]["text"]
question = "What is this news story about?"
options = [
    {"id": "world",    "description": "World: international news, politics, governments and conflicts."},
    {"id": "sports",   "description": "Sports: games, athletes, teams and competitions."},
    {"id": "business", "description": "Business: companies, markets, the economy and finance."},
    {"id": "sci_tech", "description": "Sci/Tech: science, technology, computing and space."},
]
print("state:   ", state)
print("question:", question)
for o in options:
    print(f"option:   {o['id']:<9} {o['description']}")

state:    Leaders of three neighbouring countries met in the coastal city of Port Avalen to sign a water-sharing treaty after two years of border talks.
question: What is this news story about?
option:   world     World: international news, politics, governments and conflicts.
option:   sports    Sports: games, athletes, teams and competitions.
option:   business  Business: companies, markets, the economy and finance.
option:   sci_tech  Sci/Tech: science, technology, computing and space.


## 4. The payload

The options are labelled A, B, C, D inside one prompt, and the model is asked for the
single letter. Below is the prompt, then the exact JSON body that was sent:

- `max_tokens: 1`: the model produces one token only; nothing else is generated.
- `logprobs: true`, `top_logprobs: 20`: return the scores of the 20 most likely tokens
  for that position.
- `temperature: 1.0`: leaves the model's scores as they are (we read the scores; we
  don't sample from them).

In [4]:
LETTERS = "ABCD"
lines = ["Read the text and answer the question by choosing one option.", "",
         "Text:", state, "", "Question: " + question, "", "Options:"]
lines += [f"{letter}. {o['description']}" for letter, o in zip(LETTERS, options)]
lines += ["", "Answer with the single letter of the best option and nothing else."]
prompt = "\n".join(lines)

raw = client.chat.completions.with_raw_response.create(
    model=cfg["model"],
    messages=[{"role": "user", "content": prompt}],
    max_tokens=1,
    temperature=1.0,
    logprobs=True,
    top_logprobs=20,
)
print(prompt)
print("\n--- JSON body sent to the endpoint ---")
print(json.dumps(sent["body"], indent=2))

Read the text and answer the question by choosing one option.

Text:
Leaders of three neighbouring countries met in the coastal city of Port Avalen to sign a water-sharing treaty after two years of border talks.

Question: What is this news story about?

Options:
A. World: international news, politics, governments and conflicts.
B. Sports: games, athletes, teams and competitions.
C. Business: companies, markets, the economy and finance.
D. Sci/Tech: science, technology, computing and space.

Answer with the single letter of the best option and nothing else.

--- JSON body sent to the endpoint ---
{
  "model": "Qwen/Qwen2.5-7B-Instruct",
  "messages": [
    {
      "role": "user",
      "content": "Read the text and answer the question by choosing one option.\n\nText:\nLeaders of three neighbouring countries met in the coastal city of Port Avalen to sign a water-sharing treaty after two years of border talks.\n\nQuestion: What is this news story about?\n\nOptions:\nA. World: internation

## 5. What the model sends back (trimmed)

The response is ordinary chat-completions JSON. The part that matters: the one token
the model produced, and the 20 most likely tokens with their log-probabilities (the
model's scores on a log scale; `exp(log-probability)` is the probability). Other fields
are left out here.

In [5]:
response = json.loads(raw.http_response.text)
choice = response["choices"][0]
first = choice["logprobs"]["content"][0]
trimmed = {
    "message.content": choice["message"]["content"],
    "logprobs.content[0].token": first["token"],
    "logprobs.content[0].top_logprobs": [
        {"token": t["token"], "logprob": round(t["logprob"], 4)} for t in first["top_logprobs"]],
}
print("message.content:", json.dumps(trimmed["message.content"]))
print("logprobs.content[0].token:", json.dumps(trimmed["logprobs.content[0].token"]))
print("logprobs.content[0].top_logprobs:")
for t in trimmed["logprobs.content[0].top_logprobs"]:
    print("  ", json.dumps(t))

message.content: "A"
logprobs.content[0].token: "A"
logprobs.content[0].top_logprobs:
   {"token": "A", "logprob": -0.0}
   {"token": "W", "logprob": -16.5}
   {"token": "B", "logprob": -23.875}
   {"token": "C", "logprob": -25.0}
   {"token": "D", "logprob": -25.375}
   {"token": "\u0410", "logprob": -27.5}
   {"token": "M", "logprob": -28.25}
   {"token": "P", "logprob": -28.5}
   {"token": "T", "logprob": -29.0}
   {"token": "S", "logprob": -29.125}
   {"token": "\tA", "logprob": -29.25}
   {"token": "E", "logprob": -29.25}
   {"token": "V", "logprob": -29.5}
   {"token": "F", "logprob": -29.5}
   {"token": "R", "logprob": -30.125}
   {"token": "G", "logprob": -30.25}
   {"token": "N", "logprob": -30.25}
   {"token": "Q", "logprob": -30.25}
   {"token": "L", "logprob": -30.25}
   {"token": "\uff21", "logprob": -30.5}


## 6. What we interpret

Each token that is an option letter (ignoring spaces and a trailing ".", ")" or ":")
counts toward that option; every other token is ignored. **Coverage** is the total
probability that landed on the option letters. Near 1 means the model really chose
among the options; only the top 20 tokens are returned, so a letter outside them
would be missed, and coverage below 1 would show it. The option probabilities are
then rescaled to add up to 1, and the answer is the most likely option.

In [6]:
rows = []
for t in first["top_logprobs"]:
    key = t["token"].strip().rstrip(".):").upper()
    letter = key if key in LETTERS else None
    rows.append({"token": repr(t["token"]), "logprob": round(t["logprob"], 4),
                 "probability": f"{math.exp(t['logprob']):.6f}",
                 "counts toward": options[LETTERS.index(letter)]["id"] if letter else "(ignored)"})
display(pd.DataFrame(rows).head(8))

raw_by_option = {o["id"]: 0.0 for o in options}
for t in first["top_logprobs"]:
    key = t["token"].strip().rstrip(".):").upper()
    if key in LETTERS:
        raw_by_option[options[LETTERS.index(key)]["id"]] += math.exp(t["logprob"])
coverage = sum(raw_by_option.values())
probs = {k: v / coverage for k, v in raw_by_option.items()}
answer = max(probs, key=probs.get)
print(f"coverage = {coverage:.4f}")
print("probabilities:", {k: round(v, 4) for k, v in probs.items()})
print("answer:", answer)

,token,logprob,probability,counts toward
0,'A',-0.000,1.000000,world
1,'W',-16.500,0.000000,(ignored)
2,'B',-23.875,0.000000,sports
3,'C',-25.000,0.000000,business
4,'D',-25.375,0.000000,sci_tech
5,'А',-27.500,0.000000,(ignored)
6,'M',-28.250,0.000000,(ignored)
7,'P',-28.500,0.000000,(ignored)


coverage = 1.0000
probabilities: {'world': 1.0, 'sports': 0.0, 'business': 0.0, 'sci_tech': 0.0}
answer: world


## 7. The same thing in one line

`jev/client.py` does steps 4 to 6 for you. `predict(state, question, options)` builds
the same prompt, makes the same call, and returns the answer, the probability of each
option, and the coverage.

In [7]:
from jev.client import build_prompt_text, predict

print("Same prompt as step 4:", build_prompt_text(state, question, options) == prompt)
p = predict(state, question, options, client=client, cfg=cfg)
print("answer:", p.choice, "  coverage:", round(p.coverage, 4))
pd.DataFrame({"this notebook (step 6)": probs, "predict()": p.probs}).round(4)

Same prompt as step 4: True
answer: world   coverage: 1.0


,this notebook (step 6),predict()
world,1.0,1.0
sports,0.0,0.0
business,0.0,0.0
sci_tech,0.0,0.0


## 8. All ten texts

Every text through `predict()`, next to the topic we had in mind. These are ten texts
we wrote, not a measurement of accuracy.

Raw scores from a single token are near-certain, even on the two mixed texts: there
the second choice gets far more probability than on the clear texts, but still very
little. That is why the main project rescales raw scores with a temperature fitted on
labelled data (calibration) before treating them as confidence. For a real evaluation on a labelled dataset
see `eval/run_eval.py` and the saved runs in `results/` (insurance claim notes).

In [8]:
batch = []
for t in texts:
    r = predict(t["text"], question, options, client=client, cfg=cfg)
    batch.append({"id": t["id"], "label": t["label"], "answer": r.choice,
                  "match": r.choice == t["label"],
                  "P(answer)": f"{r.probs[r.choice]:.6f}",
                  "second choice": sorted(r.probs, key=r.probs.get)[-2],
                  "P(second choice)": f"{sorted(r.probs.values())[-2]:.1e}",
                  "coverage": round(r.coverage, 4)})
table = pd.DataFrame(batch)
print(f"{table['match'].sum()} of {len(table)} match the topic we had in mind")
table

9 of 10 match the topic we had in mind


,id,label,answer,match,P(answer),second choice,P(second choice),coverage
0,text-01,world,world,True,1.000000,sports,4.3e-11,1.0
1,text-02,sports,sports,True,1.000000,business,1.8e-11,1.0
2,text-03,business,business,True,1.000000,sports,5.8e-12,1.0
3,text-04,sci_tech,sci_tech,True,1.000000,business,1.6e-11,1.0
4,text-05,world,world,True,1.000000,sports,1.5e-10,1.0
5,text-06,sports,sports,True,1.000000,business,1.6e-11,1.0
6,text-07,business,business,True,1.000000,world,2.6e-11,1.0
7,text-08,sci_tech,sci_tech,True,1.000000,business,2.3e-11,1.0
8,text-09,business,sports,False,0.999983,business,1.7e-05,1.0
9,text-10,sci_tech,sci_tech,True,0.999998,world,1.8e-06,1.0


## 9. Try your own

Change the text, the question or the options (2 to 10 of them) and run the cell below.
Nothing is trained or configured per question: the question and options travel with
each request. To see the payload and raw response for your own input, set `state`,
`question` and `options` in step 3 and re-run steps 3 to 6.

In [9]:
my_state = "The city council voted to replace its paper parking permits with a phone app built by a local startup."
my_question = "Who is most affected by this news?"
my_options = [
    {"id": "drivers",   "description": "Drivers who park in the city."},
    {"id": "startup",   "description": "The company that built the app."},
    {"id": "council",   "description": "The city council."},
]
r = predict(my_state, my_question, my_options, client=client, cfg=cfg)
print("answer:", r.choice, "  coverage:", round(r.coverage, 4))
pd.DataFrame({"probability": r.probs}).round(4)

answer: drivers   coverage: 1.0


,probability
drivers,1.0
startup,0.0
council,0.0
